# Kaggle Runner: Experiment 1

This notebook is prepared for Kaggle GPU execution. It clones the repo into the Kaggle working directory, installs the project, runs the first experiment end to end, and exports the generated artifacts as a zip you can download back into the local repo.

Before running:
- Enable Internet in the Kaggle notebook settings.
- Use a GPU notebook.
- Set `REPO_REF` to the branch or commit you want Kaggle to execute.
- If you use `meta-llama/Llama-3.1-8B-Instruct`, add a Kaggle secret named `HF_TOKEN` and make sure the model license has been accepted on Hugging Face.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/aaliyan1230/dual-direction-mech-interp.git"
REPO_REF = "main"  # branch or commit to run on Kaggle
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "dual-direction-mech-interp"

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
PROMPT_LIMIT = 200
EVAL_PROMPTS = 100
LOAD_IN_4BIT = True

RUN_CROSS_ABLATION = False
RUN_QUANTIZATION = False
RUN_CROSS_MODEL = False
GENERATE_FIGURES = True

HF_TOKEN_SECRET_NAME = "HF_TOKEN"
EXPORT_ARCHIVE = WORKDIR / "ddmi_kaggle_export.zip"

In [ ]:
def run(cmd, cwd=None, env=None):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {printable}")
    subprocess.run([str(part) for part in cmd], cwd=cwd, env=env, check=True)


def get_hf_token(secret_name: str) -> str | None:
    token = os.environ.get(secret_name)
    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return None

    try:
        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        return None


def maybe_login_to_huggingface(model_id: str):
    token = get_hf_token(HF_TOKEN_SECRET_NAME)
    needs_token = "meta-llama" in model_id.lower()
    if needs_token and not token:
        raise RuntimeError(
            f"Set the {HF_TOKEN_SECRET_NAME} Kaggle secret before running gated model downloads."
        )

    if token:
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
        print("Logged in to Hugging Face Hub.")
    else:
        print("No Hugging Face token configured; proceeding without hub login.")


def model_slug(model_id: str) -> str:
    return model_id.replace("/", "_").replace("-", "_")

In [ ]:
if REPO_DIR.exists():
    run(["rm", "-rf", REPO_DIR])

run(["git", "clone", REPO_URL, REPO_DIR])
if REPO_REF:
    run(["git", "checkout", REPO_REF], cwd=REPO_DIR)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", ".[plot]"], cwd=REPO_DIR)
if RUN_QUANTIZATION:
    run([sys.executable, "-m", "pip", "install", "-e", ".[quant]"], cwd=REPO_DIR)

maybe_login_to_huggingface(MODEL_ID)

run([
    sys.executable,
    "-c",
    "import torch; print({'cuda_available': torch.cuda.is_available(), 'device_count': torch.cuda.device_count()})",
])

In [ ]:
safe_name = model_slug(MODEL_ID)
direction_dir = REPO_DIR / "artifacts" / "directions"
safety_artifact = direction_dir / f"{safe_name}_safety.json"
epistemic_artifact = direction_dir / f"{safe_name}_epistemic.json"
comparison_artifact = direction_dir / f"{safe_name}_comparison.json"

common_flags = ["--model-id", MODEL_ID, "--prompt-limit", str(PROMPT_LIMIT)]
if LOAD_IN_4BIT:
    common_flags.append("--load-in-4bit")

run([
    sys.executable,
    "scripts/extract_directions.py",
    *common_flags,
    "--direction-type",
    "safety",
    "--output",
    str(safety_artifact),
], cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/extract_directions.py",
    *common_flags,
    "--direction-type",
    "epistemic",
    "--output",
    str(epistemic_artifact),
], cwd=REPO_DIR)

run([
    sys.executable,
    "scripts/compare_directions.py",
    "--safety-artifact",
    str(safety_artifact),
    "--epistemic-artifact",
    str(epistemic_artifact),
    "--output",
    str(comparison_artifact),
], cwd=REPO_DIR)

print({
    "safety_artifact": str(safety_artifact),
    "epistemic_artifact": str(epistemic_artifact),
    "comparison_artifact": str(comparison_artifact),
})

In [ ]:
if RUN_CROSS_ABLATION:
    ablation_cmd = [
        sys.executable,
        "scripts/cross_ablation.py",
        "--model-id",
        MODEL_ID,
        "--safety-direction",
        str(safety_artifact),
        "--epistemic-direction",
        str(epistemic_artifact),
        "--output",
        str(REPO_DIR / "artifacts" / "cross_ablation" / f"{safe_name}_results.json"),
        "--eval-prompts",
        str(EVAL_PROMPTS),
    ]
    if LOAD_IN_4BIT:
        ablation_cmd.append("--load-in-4bit")
    run(ablation_cmd, cwd=REPO_DIR)

if RUN_QUANTIZATION:
    run([
        sys.executable,
        "scripts/quantization_sweep.py",
        "--model-id",
        MODEL_ID,
        "--fp16-safety-artifact",
        str(safety_artifact),
        "--output",
        str(REPO_DIR / "artifacts" / "quantization" / f"{safe_name}_sweep.json"),
        "--prompt-limit",
        str(min(PROMPT_LIMIT, 100)),
    ], cwd=REPO_DIR)

if RUN_CROSS_MODEL:
    cross_model_cmd = [
        sys.executable,
        "scripts/cross_model_replication.py",
        "--output-dir",
        str(REPO_DIR / "artifacts" / "cross_model"),
        "--prompt-limit",
        str(PROMPT_LIMIT),
    ]
    if LOAD_IN_4BIT:
        cross_model_cmd.append("--load-in-4bit")
    run(cross_model_cmd, cwd=REPO_DIR)

if GENERATE_FIGURES:
    run([
        sys.executable,
        "scripts/generate_figures.py",
        "--artifacts-dir",
        "artifacts",
        "--output-dir",
        "artifacts/figures",
    ], cwd=REPO_DIR)

In [ ]:
run([
    sys.executable,
    "scripts/export_artifacts_bundle.py",
    "--output",
    str(EXPORT_ARCHIVE),
    "--label",
    safe_name,
    "--include-path",
    "README.md",
], cwd=REPO_DIR)

print(f"Download this file from the Kaggle Output pane: {EXPORT_ARCHIVE}")
for path in sorted((REPO_DIR / "artifacts").rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_DIR))

## What This Produces

By default this notebook runs the first headline experiment only:
- Safety direction extraction
- Epistemic direction extraction
- Direction comparison
- Figure generation when possible
- Export of bundled outputs to `/kaggle/working/ddmi_kaggle_export.zip`

Once Kaggle finishes, download that zip from the Output pane and extract it into the local repo before reviewing and committing artifacts.